In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6KURKawf3E3dL507AF_b154Bgohi_niJbY3S7Itel-N2A"

In [ ]:
!pip install -q --upgrade youtube-transcript-api langchain-community langchain-google-genai faiss-cpu python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

**step1 : INdexing (document ingestion)**

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "4Vz6L8B73i4"

try:
    # Create an instance
    ytt_api = YouTubeTranscriptApi()

    # Fetch transcript
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    # Convert to plain text
    transcript = " ".join(
        snippet.text for snippet in transcript_list
    )

    print(transcript)
    print(len(transcript))

except TranscriptsDisabled:
    print("NO caption available for this video")

If you let people have their phones out, the IQ of your employees is lower. Cuz there was a study done when the phone was visible to you. Scores were significantly decreased. >> Like my attention decreased. >> Yes, Raj. It was also general fluid intelligence. That means you're dumber. This is the scary reality of how attention works in the brain. people who hold their phone or put their phone out on the table when they're talking to people and they first meet you. You seem less trustworthy, less capable, less intelligent, all of these different things. [music] I'm a cognitive neuroscientist. Study how we can train the brain to be better. Every single time you [music] switch your attention, you pay for it in time and you pay for it in energy. And if you don't believe me, we can test it out. >> Grab a piece of paper. You have some paper and a pen. Now, on the top line, [music] I want you to write, "I am a great multitasker." And then on the bottom line, I want you to write the numbers 1 

chunks split

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

206

In [ ]:
!pip install -q -U langchain-huggingface sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 27.2 MB/s eta 0:00:00


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Local embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector store
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Vector store created successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully!


In [ ]:
vector_store.index_to_docstore_id

{0: 'e96e9bee-2e37-4d4c-9dd3-90f44fd34ac8',
 1: '86812447-df9c-4573-a503-9307956794ea',
 2: '08170605-deaa-4fce-996a-737cf4f28afe',
 3: '9c169e87-bf29-46ab-84e0-8db367f04493',
 4: '7eacac1d-e473-44a4-b116-2af5ac92d89d',
 5: 'adbcd400-fd9a-4a44-898e-701b46d8559e',
 6: '161d1401-6065-4895-b341-548a7b9dc298',
 7: '0f36dac0-b53a-47d3-90ee-4acfca6469d3',
 8: '844630c9-3730-4aee-ae43-9d70c6037aa6',
 9: '227123d1-9cf3-4b03-b4ba-5bc0d4b324c5',
 10: 'e4b8b88a-c23a-423d-9785-be16ed4bd036',
 11: '9d6d4d98-06c0-4453-8da1-c6d6825ab0db',
 12: '437398fb-31bb-4dd6-b06a-44b8ebb207e3',
 13: 'c2fb84ff-3e9d-4f3b-8e23-d4b45119dd85',
 14: 'c430f2e5-f66d-42ac-aedc-20b804020ed2',
 15: '2fb9b86b-fc67-4724-8d57-73002a67557d',
 16: '6a4bb577-d1a9-4912-ad73-bdeee436abf4',
 17: 'ef5fe0fa-4d64-4ff5-90fd-2c3e46fb4097',
 18: '7ab337ed-3ab1-418b-ae68-d3a3e7e06991',
 19: 'd92dadc2-befd-45ad-a1d5-c18ff8878801',
 20: '26c0ee5d-513e-4810-807c-f30a78de482a',
 21: 'b8c6b1d7-bd63-4af4-a437-b3a797bf5996',
 22: 'fec3c3a1-2049-

In [ ]:
retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})


In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7c62f239c590>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke('What is memory')

[Document(id='6326cc66-d294-4b24-b3ca-31482a4375e0', metadata={}, page_content="memory then of the positive things? M in general improving your memory and this is actually a a I think kind of a fun fact that's a little bit um counterintuitive. >> If you want to strengthen your memory, open the aperture of your attention if you you can only remember what you're attending to, right? You can't remember something that you weren't paying attention to. >> Yeah. So if you can heighten and sharpen your focus and your attention, then your memory will get better. It's like the funnel, right? The attention is what you're paying attention to. All the stuff that's getting in. So if you want more stuff to get in, focus on your attention if you want to remember more stuff, right? >> Things h you have to come in in order to go down into memory. >> Yeah. But >> that's pretty much the end of the brain city map. One thing that I keep failing at is waking up super early. And I've tried so many times. Some

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.2
)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of focus discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='3ec463ba-21c1-439d-a4b1-07bbccb967e5', metadata={}, page_content='So going back to the example, you sit down and you have a goal. >> H >> you have that\'s your meaning. >> Okay, >> your CEO, your CEN, but it\'s your CEO. Let\'s say that\'s going to start focusing. >> Okay. >> Okay. Your DN says, "Got it. No problem. Let\'s focus on the computer. Let\'s focus on the research." So your DA, the watchtowwer goes, it\'s focusing on the work. As long as you keep focusing and as long as nothing distracts you or interrupts you, then maybe as long as you can, you will keep focusing. >> This is my tension. >> Yeah. But what\'s the usual setup? Remember when we talked about your phone and where you keep it? >> It\'s like >> here >> there\'s one notification, another there are five tabs opening, two people talking, and then I have to be everywhere and then I have to message >> probably scroll a little in between. That\'s what always happens. So now you\'ve got every time the notifica

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'So going back to the example, you sit down and you have a goal. >> H >> you have that\'s your meaning. >> Okay, >> your CEO, your CEN, but it\'s your CEO. Let\'s say that\'s going to start focusing. >> Okay. >> Okay. Your DN says, "Got it. No problem. Let\'s focus on the computer. Let\'s focus on the research." So your DA, the watchtowwer goes, it\'s focusing on the work. As long as you keep focusing and as long as nothing distracts you or interrupts you, then maybe as long as you can, you will keep focusing. >> This is my tension. >> Yeah. But what\'s the usual setup? Remember when we talked about your phone and where you keep it? >> It\'s like >> here >> there\'s one notification, another there are five tabs opening, two people talking, and then I have to be everywhere and then I have to message >> probably scroll a little in between. That\'s what always happens. So now you\'ve got every time the notification comes in, salance network goes, "Hey, look over here. What\'s over there? 

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don\'t know.\n\n      So going back to the example, you sit down and you have a goal. >> H >> you have that\'s your meaning. >> Okay, >> your CEO, your CEN, but it\'s your CEO. Let\'s say that\'s going to start focusing. >> Okay. >> Okay. Your DN says, "Got it. No problem. Let\'s focus on the computer. Let\'s focus on the research." So your DA, the watchtowwer goes, it\'s focusing on the work. As long as you keep focusing and as long as nothing distracts you or interrupts you, then maybe as long as you can, you will keep focusing. >> This is my tension. >> Yeah. But what\'s the usual setup? Remember when we talked about your phone and where you keep it? >> It\'s like >> here >> there\'s one notification, another there are five tabs opening, two people talking, and then I have to be everywhere and then I have to message

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Yes, the topic of focus is discussed in the transcript. Here is what was discussed:\n\n* **Brain Networks and Focus:**\n  * **CEN (Central Executive Network / frontal parietal area):** Responsible for goal-directed attention. It tries to quiet everything else down so you can stay focused on achieving a goal (e.g., doing research on a computer).\n  * **DN (Dorsal Attention Network):** Acts like a "watchtower" or "spotlight/flashlight" of attention, pointing focus toward whatever task needs attention.\n\n* **Distractions and Interruptions:**\n  * Keeping a focus state is interrupted by distractions like phone notifications, multiple open tabs, or people talking, which cause the salience network or "alarm tower" to direct your attention away from your work (e.g., leaving a Google doc to handle an emergency or notification).\n\n* **Unfocused Brains:**\n  * The context mentions that people/the next generation are being trained to be so distracted and unfocused tha

building chains

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('What is memory')

{'context': 'memory then of the positive things? M in general improving your memory and this is actually a a I think kind of a fun fact that\'s a little bit um counterintuitive. >> If you want to strengthen your memory, open the aperture of your attention if you you can only remember what you\'re attending to, right? You can\'t remember something that you weren\'t paying attention to. >> Yeah. So if you can heighten and sharpen your focus and your attention, then your memory will get better. It\'s like the funnel, right? The attention is what you\'re paying attention to. All the stuff that\'s getting in. So if you want more stuff to get in, focus on your attention if you want to remember more stuff, right? >> Things h you have to come in in order to go down into memory. >> Yeah. But >> that\'s pretty much the end of the brain city map. One thing that I keep failing at is waking up super early. And I\'ve tried so many times. Some like I I\'ve somehow what do you [clears throat] say? Som

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided transcript, the text discusses the following main topics:\n\n* **Long-Form vs. Short-Form Media:** Long-form media (such as long podcasts, educational YouTube videos, and movies) challenges the mind, helps people learn, improves memory, and increases empathy. In contrast, short-form media and "doom scrolling" make people\'s memory worse, reduce empathy, and are quickly forgotten (such as not being able to remember a short video watched just a few reels ago).\n* **Netflix and "Second Screen Viewing":** Netflix asks creators if shows are suitable for "second screen viewing," designing content with quick scenes to engage viewers who are simultaneously on their phones.\n* **Podcast Wrap-Up:** The speakers conclude a podcast episode about the brain, productivity, and cognition, teasing a future episode about "10 laws" for becoming a "superhuman," and emphasizing a focus on "impact maximization" over fun for the next few years.'